# dev/phase — run one phase, from the project itself

Every phased project gets this notebook unchanged. It types nothing
about the protocol: `Bench` reads `launch.yaml` and loads the same
scene, recipes, actions, parameters, checks and phase list `main.py`
runs, so a green cell here is the run's own code passing.

`bench.phase(name, **kwargs)` seeds the closure facts of every earlier
phase, puts the model where those phases leave things (`Phase.layout`),
then plans and executes that ONE phase and stops. The planner orders the
actions exactly as a real run would. `kwargs` are the operator's — what
the HMI's Start would send (here `batch_size`).

Run phases back to back (1, then 2) and the second starts from where the
first left the bench. Jump straight to a later one and the cell prints
what it assumed — set the real bench the same way first.

# Start / kill the notebook viewer

In [ ]:
# ── START the notebook 3D-viewer server on port 8000 (idempotent) ──
# Port 5000 belongs to the orchestrator (gui/server.py) — that one does
# NOT stream the viewer; workspace/server.py does.
import subprocess, urllib.request, time, os
from pathlib import Path

def viewer_up():
    try:
        urllib.request.urlopen('http://127.0.0.1:8000/', timeout=2)
        return True
    except Exception:
        return False

if not viewer_up():
    srv = Path.home() / 'Downloads/workspace/workspace/server.py'
    try:
        log = open(f'/tmp/workspace_viewer.{os.getuid()}.log', 'ab')
    except OSError:
        log = subprocess.DEVNULL
    subprocess.Popen(['sudo', 'env', 'PORT=8000', 'python3', str(srv)],
                     cwd=srv.parent, stdout=log, stderr=log,
                     start_new_session=True)   # survives kernel restarts
    while not viewer_up():
        time.sleep(0.5)
import socket as _sk
_s=_sk.socket(_sk.AF_INET,_sk.SOCK_DGRAM)
try: _s.connect(('8.8.8.8',80)); _ip=_s.getsockname()[0]
except OSError: _ip='localhost'
finally: _s.close()
print(f'3D viewer: http://{_ip}:8000/' if viewer_up() else 'viewer down')

In [ ]:
# ── KILL the notebook viewer server (frees port 8000) ──
!sudo fuser -k 8000/tcp 2>/dev/null || echo 'port 8000 already free'

## The project

In [ ]:
from pathlib import Path
from workspace.bt import Bench

# Walk up to the folder holding launch.yaml — scene, recipes, actions,
# parameters, checks and the phase list all come from there. Nothing
# about the protocol is typed in this notebook.
PROJ = Path.cwd()
while not (PROJ / 'launch.yaml').exists():
    PROJ = PROJ.parent

bench = Bench(PROJ, port=8000)   # port 8000 — the notebook's own viewer
print('phases:', [p.name for p in bench._phases(bench.kwargs())])

## Prepare the phase

Puts the MODEL where the phase starts — the viewer shows it — and prints every item it moved. Nothing moves on the robot. Set the real bench to match, then choose sim or real below and run.


In [ ]:
bench.prepare('weighed_2', batch_size=2)

## Simulation on or off

`True` -> SimulationAPI, no hardware. `False` -> the real robot.

In [ ]:
bench.core.simulation(True)

## Robot up — motors on, rail homed

The project's own `Start` action, run through the bench: motors on, rail homed (an already-homed rail and sim short-circuit), park. `phase()` runs it again at its start — harmless — so this cell is for driving actions by hand with `bench.run`, or for homing before `prepare`.


In [ ]:
from actions import Start
bench.run(Start)

## Run a phase

Any phase, any order. Change the name and run the cell again.

In [ ]:
bench.phase('weighed_2', batch_size=2)

## Back to the launch scene

Phases run back to back continue from where the last one left the model, tool included — nothing is reset behind your back. Before running an EARLIER phase again, snap the model back and set the real bench the same way.

In [ ]:
bench.reset()

## One action by hand

The building block underneath: a project action class, executed with the project's context — checks, tool swap and effects as in a run. Facts accumulate across calls.

In [ ]:
from actions import Start
from actions.phase_1 import Pick1
bench.run(Start)
bench.run(Pick1, 0)
sorted(bench.facts)